In [ ]:
# Run once in a fresh notebook environment.
%pip install -q numpy scipy pandas matplotlib


# Round 5 Reproduction

**Purpose.** Reproduce the four Round 5 analyses: full-information objective diagnosis, value of repeated information, independent active-search confirmation, and the frozen non-myopic DP comparison.

The notebook uses the source-controlled manifest workflow. It does not contain a separate implementation of the model.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the repository checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print(PROJECT_ROOT)


In [ ]:
RUN = False
FAMILY = "six_sample"  # oracle, six_sample, custom_rr, or fixed_budget
CONFIGS_JSON = None     # required for custom_rr and fixed_budget
OUTPUT_DIR = "results/round_05_notebook/" + FAMILY
EPISODES = 1200
EPISODES_PER_TASK = 5
VOI_SAMPLES = 500
SEED_NAMESPACE_OFFSET = 0
SAMPLE_BUDGETS = "0,2,4,6,8,10,12"
print({"run": RUN, "family": FAMILY, "episodes": EPISODES, "episodes_per_task": EPISODES_PER_TASK, "voi_samples": VOI_SAMPLES})


## Freeze the analysis manifest

**Test purpose.** Record the environment definitions, code commit, seed namespace, solver settings, and shard boundaries before evaluating episodes.


In [ ]:
create_command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "active_search_evaluation_workflow.py"),
    "create",
    "--family", FAMILY,
    "--output-dir", OUTPUT_DIR,
    "--episodes", str(EPISODES),
    "--episodes-per-task", str(EPISODES_PER_TASK),
    "--observation-draws", str(VOI_SAMPLES),
    "--seed-namespace-offset", str(SEED_NAMESPACE_OFFSET),
    "--sample-budgets", SAMPLE_BUDGETS,
]
if CONFIGS_JSON:
    create_command.extend(["--configs-json", CONFIGS_JSON])
print(" ".join(create_command))
if RUN:
    subprocess.run(create_command, cwd=PROJECT_ROOT, check=True)


## Execute and strictly collect tasks

This cell is server-agnostic. A compute environment may distribute the frozen task indices in any way, provided every task uses the same manifest. The direct loop below is suitable only when sequential execution is intended.


In [ ]:
manifest_path = PROJECT_ROOT / OUTPUT_DIR / "active_search_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print({"manifest": str(manifest_path), "task_count": len(manifest["tasks"])})
    if RUN:
        for task in manifest["tasks"]:
            subprocess.run([
                sys.executable, str(PROJECT_ROOT / "scripts" / "active_search_evaluation_workflow.py"),
                "run-task", "--manifest", str(manifest_path), "--task-index", str(task["task_index"]),
            ], cwd=PROJECT_ROOT, check=True)
        subprocess.run([
            sys.executable, str(PROJECT_ROOT / "scripts" / "active_search_evaluation_workflow.py"),
            "collect", "--manifest", str(manifest_path),
        ], cwd=PROJECT_ROOT, check=True)
else:
    print("Create the manifest first.")


## Generate the validated report

Point these paths to the strictly collected oracle, objective analysis, fixed-budget, discovery, confirmation, and solver outputs. The report generator checks expected task and episode counts before producing its summary.


In [ ]:
GENERATE_REPORT = False
REPORT_INPUTS = {
    "oracle-dir": "results/active_search_oracle_full",
    "oracle-analysis-dir": "results/active_search_oracle_analysis",
    "formal-dir": "results/active_search_formal_summaries",
    "discovery-dir": "results/active_search_six_sample_discovery",
    "confirmation-dir": "results/active_search_six_sample_confirmation",
    "solver-dir": "results/active_search_solver_comparison",
    "output-dir": "results/active_search_report",
}
report_command = [sys.executable, str(PROJECT_ROOT / "scripts" / "generate_active_search_report.py")]
for key, value in REPORT_INPUTS.items():
    report_command.extend(["--" + key, value])
print(" ".join(report_command))
if GENERATE_REPORT:
    subprocess.run(report_command, cwd=PROJECT_ROOT, check=True)


In [ ]:
import pandas as pd

report_dir = PROJECT_ROOT / REPORT_INPUTS["output-dir"]
summary_path = report_dir / "supporting_data" / "active_search_report_summary.json"
confirmation_path = report_dir / "supporting_data" / "active_search_confirmation_comparison.csv"
if summary_path.exists():
    display(json.loads(summary_path.read_text()))
if confirmation_path.exists():
    display(pd.read_csv(confirmation_path))
